# Root Motion: Small Auxiliary Translation Predictor

The main VQ-VAE (`02_train_bones.ipynb`) trains on the full `bones` feature (root xyz + unit bone directions) and is unaffected by this notebook -- its representation and training are unchanged.

**Why this exists**: on a real checkpoint, root/translation reconstruction showed frame-to-frame velocity error 3.1x larger than the ground truth's own natural velocity magnitude -- visible jitter -- while bone-direction (pose) reconstruction was good. The clearest cause: velocity error right at `chunk_len` boundaries was ~4x worse than mid-chunk, since each chunk is encoded/decoded by the main VQ-VAE completely independently (no cross-chunk continuity).

**Fix**: a small, separate model predicts root motion directly from the (already well-reconstructed) bone-direction sequence, run on **full, un-chunked sequences** -- so there's no chunk-boundary discontinuity for this signal at all. It never touches vector quantization; it's a small continuous regressor (`src/model/root_motion_net.py`, ~129K params vs. the main VQ-VAE's ~21.6M).

**Representation** (`src/data/root_motion.py`), HumanML3D-style root decomposition instead of absolute per-frame position:
- `heading_vel` -- wrapped angular difference of the character's facing direction (derived from the L_hip->R_hip vector, rotated 90 degrees), radians/frame.
- `local_vel_x`, `local_vel_z` -- root's horizontal (ground-plane) velocity, canonicalized into the character's *own* current-heading local frame (not raw world-frame X/Z).
- `height` -- root's absolute Y (not integrated -- height doesn't drift the way ground-plane position does over an integrated trajectory).

Reconstruction integrates (cumulative sum) `heading_vel` and the rotated-back `local_vel_{x,z}` starting from an initial position/heading; `height` is used directly per frame. Round-trip verified on 20 real test sequences: 19/20 sequences show 1e-5 to 3e-4 m error (float-precision level); one outlier at 3mm from a near-degenerate hip-vector projection during a fast turn (heading becomes ill-conditioned when the hip vector's ground-plane projection shrinks toward zero). All far below the ~0.6m error this is meant to fix.

**Input to the small net**: bone-direction unit vectors only (63-dim, body-only skeleton) -- not the main VQ-VAE's root_xyz channel, which is exactly the jittery signal being replaced.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.splits import load_splits
from src.data.normalize import NormStats, compute_stats
from src.data.root_motion import extract_root_motion, integrate_root_motion
from src.data.root_motion_dataset import RootMotionDataset, collate_root_motion
from src.model.root_motion_net import RootMotionNet
from src.skeleton import NUM_BODY_JOINTS
from src.train_root_motion import train_root_motion


## Data setup

In [2]:
ROOT = Path.cwd().parent
raw_dir = ROOT / "data/raw/motion_text_7000"
splits = load_splits(ROOT / "data/splits")

stats_root_motion = NormStats.load(ROOT / "data/stats/stats_root_motion.npz")
print("root motion stats -- mean:", stats_root_motion.mean, "std:", stats_root_motion.std)

train_ds = RootMotionDataset(splits["train"], raw_dir, root_motion_stats=stats_root_motion)
val_ds = RootMotionDataset(splits["val"], raw_dir, root_motion_stats=stats_root_motion)
test_ds = RootMotionDataset(splits["test"], raw_dir, root_motion_stats=stats_root_motion)

print(f"train: {len(train_ds)} sequences (full-length, no chunking)")
print(f"val:   {len(val_ds)} sequences")
print(f"test:  {len(test_ds)} sequences")


root motion stats -- mean: [-5.7888136e-04 -9.4131716e-03 -3.3367053e-04  9.0395021e-01] std: [0.03556456 0.02075223 0.00927879 0.17847876]
train: 5408 sequences (full-length, no chunking)
val:   300 sequences
test:  301 sequences


In [3]:
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_root_motion)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_root_motion)

bone_dirs, root_motion, mask = next(iter(train_loader))
print("bone_dirs:", bone_dirs.shape, "root_motion:", root_motion.shape, "mask:", mask.shape)


bone_dirs: torch.Size([32, 200, 63]) root_motion: torch.Size([32, 199, 4]) mask: torch.Size([32, 199])


## Model setup

In [4]:
device = torch.device("cpu")
print("device:", device)

model = RootMotionNet(input_dim=63, hidden=64, out_dim=4)
print("num params:", sum(p.numel() for p in model.parameters()))


device: cpu
num params: 129028


## Training

In [8]:
EPOCHS = 30
LR = 1e-3

history = train_root_motion(model, train_loader, val_loader, device, epochs=EPOCHS, lr=LR, log_every=5)

checkpoint_path = ROOT / "checkpoints/root_motion_net.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print("saved to", checkpoint_path)


epoch    0 | train_l1=0.17903 val_l1=0.19620
epoch    5 | train_l1=0.14974 val_l1=0.16687
epoch   10 | train_l1=0.13339 val_l1=0.14974
epoch   15 | train_l1=0.12282 val_l1=0.14243
epoch   20 | train_l1=0.11596 val_l1=0.13521
epoch   25 | train_l1=0.10853 val_l1=0.12473
saved to /Users/bereket/Desktop/Jupyter/metrina-vqvae/checkpoints/root_motion_net.pt


## Reconstruction check: integrated trajectory vs. ground truth

In [6]:
model.eval()

def eval_sequence(sid: str):
    joints = np.load(raw_dir / "joints" / f"{sid}.npy").astype(np.float32)[:, :NUM_BODY_JOINTS, :]

    from src.fk import joints_to_bones
    from src.skeleton import BODY_PARENTS

    _, bone_vecs = joints_to_bones(joints, parents=BODY_PARENTS, num_joints=NUM_BODY_JOINTS)
    lengths = np.linalg.norm(bone_vecs, axis=-1, keepdims=True)
    unit_dirs = (bone_vecs / np.clip(lengths, 1e-8, None)).reshape(joints.shape[0], -1)

    with torch.no_grad():
        x = torch.from_numpy(unit_dirs.astype(np.float32)).unsqueeze(0)
        pred_norm = model(x)[0].numpy()
    pred = stats_root_motion.denormalize(pred_norm)

    from src.data.root_motion import _heading
    init_heading = _heading(joints[:1])[0]
    recon_root = integrate_root_motion(pred, init_xz=tuple(joints[0, 0, [0, 2]]), init_heading=init_heading)

    gt_root = joints[:, 0, :]
    mpjpe_mm = np.linalg.norm(recon_root - gt_root, axis=-1).mean() * 1000

    gt_vel = np.linalg.norm(gt_root[1:] - gt_root[:-1], axis=-1)
    recon_vel = np.linalg.norm(recon_root[1:] - recon_root[:-1], axis=-1)
    vel_err = np.abs(recon_vel - gt_vel).mean()

    return mpjpe_mm, vel_err

for sid in splits["test"][:10]:
    mpjpe_mm, vel_err = eval_sequence(sid)
    print(f"{sid}: root MPJPE={mpjpe_mm:.1f}mm, |velocity error|={vel_err:.4f}m/frame")


002591: root MPJPE=277.0mm, |velocity error|=0.0064m/frame
000972: root MPJPE=116.0mm, |velocity error|=0.0036m/frame
006482: root MPJPE=68.5mm, |velocity error|=0.0012m/frame
003412: root MPJPE=49.1mm, |velocity error|=0.0009m/frame
007740: root MPJPE=126.6mm, |velocity error|=0.0012m/frame
007811: root MPJPE=108.3mm, |velocity error|=0.0018m/frame
006567: root MPJPE=109.1mm, |velocity error|=0.0038m/frame
002620: root MPJPE=10.1mm, |velocity error|=0.0005m/frame
003813: root MPJPE=78.5mm, |velocity error|=0.0009m/frame
001515: root MPJPE=390.6mm, |velocity error|=0.0106m/frame
